In [1]:
append_str = '_stickfunction5vis'

In [2]:
import os
import sys
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import seaborn as sns
from nilearn.masking import apply_mask

In [3]:
data_dir = "/data/pt_02747/action_hippo/data/derivatives/"

# subjects are all folders in beta_dir
subs = os.listdir(data_dir)
subs = [s for s in subs if s.startswith("sub-")]
subs = [s for s in subs if not s.endswith("html")]
subs.sort()

In [4]:
regions = ['Entorhinal-left', 'Entorhinal-right']
# replace - with _ for region name

threshold = 0.2

In [5]:
for region in regions:
    region_name = region.replace('-', '_')
    for sub in subs:
        roi_dir = os.path.join(data_dir, sub, "anat", "roi")
        reliable_map = os.path.join(roi_dir, "voxel_reliability", f"{sub}_space-T1w{append_str}_pvals_euclidean_reliability.nii.gz")

        # load reliability map if it exists; if not, continue

        if os.path.exists(reliable_map):
            # load map
            reliable_map = nib.load(reliable_map)
        else:
            continue

        # threshold map at p < 0.5 to create mask
        reliable_map_data = reliable_map.get_fdata()

        # create mask
        reliable_map_mask = np.zeros(reliable_map_data.shape)
        reliable_map_mask[reliable_map_data < threshold] = 1

        # turn into nifti mask
        reliable_map_mask = nib.Nifti1Image(reliable_map_mask, reliable_map.affine, reliable_map.header)

        # get specific region roi
        roi = os.path.join(roi_dir, region, f"{sub}_mask-{region_name}.nii.gz")

        # load roi
        roi = nib.load(roi)

        # intersect roi with mask using *
        roi_data = roi.get_fdata() * reliable_map_mask.get_fdata()


        # turn back into nii file
        roi = nib.Nifti1Image(roi_data, roi.affine, roi.header)

        # save roi

        # create output directory 
        region_path_reliable = 'rel2' + region
        roi_dir_reliable = os.path.join(roi_dir, region_path_reliable)
        if not os.path.exists(roi_dir_reliable):
            os.makedirs(roi_dir_reliable)

        # replace - with _ for region name
        region_name_reliable = region_path_reliable.replace('-', '_')

        # save roi
        nib.save(roi, os.path.join(roi_dir_reliable, f"{sub}_mask-{region_name_reliable}{append_str}.nii.gz"))